# Time-to-Resolution Robustness — Literature-Motivated Main Specification

This notebook reruns the horizon analysis using the market-specific models built from the corrected YES/NO trading dataset.

The main Enhanced specification now contains:
- probability level;
- one-day probability innovation;
- probability × time-to-resolution;
- cumulative supportive trading volume;
- net support/opposition imbalance.

Traditional ML additionally includes `Days_to_Resolution_Years`.

Cumulative probability change and volume surge remain RQ2/robustness variables rather than headline predictors.

The purpose is to test whether the incremental value of the literature-motivated Polymarket information changes as the contract approaches resolution.


## Step 1 — Upload the Required Files

Upload:

1. `Final_Competitive_Markets_MODEL_READY_with_CORRECTED_PM_Trading_Features.xlsx`
2. `Market_Specific_Robustness_All_Models_Results.xlsx`

The second file must be generated by the **new corrected market-specific robustness notebook** before running this notebook.


In [ ]:
# ============================================================
# STEP 1: Upload Required Files
# ============================================================

from google.colab import files

uploaded = files.upload()

print("Uploaded files:")
for file_name in uploaded.keys():
    print(" -", file_name)


## Step 2 — Install and Import Packages


In [ ]:
# ============================================================
# STEP 2: Install and Import Packages
# ============================================================

!pip -q install xgboost openpyxl statsmodels scikit-learn scipy

import ast
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import softmax
from scipy.stats import spearmanr

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit

from xgboost import XGBClassifier

from statsmodels.discrete.conditional_models import ConditionalLogit
import statsmodels.api as sm

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 200)

print("Packages imported successfully.")


## Step 3 — Load the Model-Ready Data and Previous Robustness Results


In [ ]:
# ============================================================
# STEP 3: Load Corrected Dataset + New Market-Specific Results
# ============================================================

import glob
import os

data_candidates = [
    x for x in glob.glob("/content/*.xlsx")
    if "CORRECTED_PM_Trading_Features" in os.path.basename(x)
    and "MODEL_READY" in os.path.basename(x).upper()
]

results_candidates = [
    x for x in glob.glob("/content/*.xlsx")
    if "Market_Specific_Robustness_All_Models_Results" in os.path.basename(x)
]

if len(data_candidates) != 1:
    raise ValueError(
        f"Could not uniquely detect the corrected dataset: {data_candidates}"
    )

if len(results_candidates) != 1:
    raise ValueError(
        f"Could not uniquely detect the NEW market-specific results: {results_candidates}"
    )

DATA_PATH = data_candidates[0]
RESULTS_PATH = results_candidates[0]

df = pd.read_excel(
    DATA_PATH,
    sheet_name="Model_Ready_Master"
)

best_ml = pd.read_excel(
    RESULTS_PATH,
    sheet_name="Best_ML_Params"
)

saved_final_predictions = pd.read_excel(
    RESULTS_PATH,
    sheet_name="Final_Event_Predictions"
)

for col in [
    "Date",
    "Next_Financial_Date",
    "Financial_Prev1_Date",
    "Financial_Prev2_Date",
    "Financial_Prev3_Date",
    "PM_Signal_Date_Lag1",
    "PM_Refined_Activity_Signal_Date_Lag1"
]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

if "Date" in saved_final_predictions.columns:
    saved_final_predictions["Date"] = pd.to_datetime(
        saved_final_predictions["Date"]
    )

df = df.sort_values(
    ["Market_ID", "Date", "Event_ID", "Asset"]
).reset_index(drop=True)

print("Dataset:", DATA_PATH)
print("Results:", RESULTS_PATH)
print("Model-ready rows:", len(df))
print("Model-ready events:", df["Event_ID"].nunique())
print("Saved final prediction rows:", len(saved_final_predictions))
print("Selected ML parameter rows:", len(best_ml))

print("\nEvents by market:")
print(df.groupby("Market_ID")["Event_ID"].nunique())

display(best_ml.head())


## Step 4 — Define the Existing Feature Sets

No new predictors are introduced.


In [ ]:
BASELINE_FEATURES = [
    "Lag1_Return",
    "Lag2_Return",
    "Lag3_Return"
]

PM_MAIN_FEATURES = [
    "PM_Prob_Lag1",
    "PM_Change_1D_Lag1",
    "PM_Prob_x_TimeToResolution",
    "PM_Log_Cumulative_Support_Notional_Lag1",
    "PM_Net_Support_Imbalance_Lag1"
]

CL_ENHANCED_FEATURES = (
    BASELINE_FEATURES
    + PM_MAIN_FEATURES
)

ML_ENHANCED_FEATURES = (
    CL_ENHANCED_FEATURES
    + ["Days_to_Resolution_Years"]
)

MARKETS = [
    "BTC_GOLD_2025",
    "BTC_GOLD_2026",
    "BTC_GOLD_SP500"
]

ML_MODELS = [
    "Logistic_Regression",
    "Random_Forest",
    "XGBoost"
]

ALL_MODELS = [
    "Conditional_Logit",
    "Logistic_Regression",
    "Random_Forest",
    "XGBoost"
]

feature_set_table = pd.DataFrame([
    {
        "Specification": "Baseline",
        "Model_Family": "All",
        "Features": " + ".join(BASELINE_FEATURES)
    },
    {
        "Specification": "Enhanced",
        "Model_Family": "Conditional Logit",
        "Features": " + ".join(CL_ENHANCED_FEATURES)
    },
    {
        "Specification": "Enhanced",
        "Model_Family": "Traditional ML",
        "Features": " + ".join(ML_ENHANCED_FEATURES)
    }
])

assert (
    df["PM_Refined_Activity_Signal_Date_Lag1"] < df["Date"]
).all(), "Corrected PM activity timing leakage detected."

required = list(dict.fromkeys(
    BASELINE_FEATURES
    + CL_ENHANCED_FEATURES
    + ML_ENHANCED_FEATURES
))

assert df[required].notna().all().all(), (
    "Missing values detected in the main time-to-resolution model features."
)

print("CL Enhanced:", CL_ENHANCED_FEATURES)
print("\nML Enhanced:", ML_ENHANCED_FEATURES)


## Step 5 — Recreate the Exact Market-Specific 80/20 Splits


In [ ]:
# ============================================================
# STEP 5: Recreate Market-Specific Development / Test Splits
# ============================================================

market_splits = {}

for market_id in MARKETS:

    market_df = df[
        df["Market_ID"] == market_id
    ].copy()

    dates = np.array(
        sorted(market_df["Date"].unique())
    )

    cut = int(np.floor(0.80 * len(dates)))

    market_splits[market_id] = {
        "Dev_Dates": dates[:cut],
        "Test_Dates": dates[cut:]
    }

    print("\n", market_id)
    print("Total dates:", len(dates))
    print("Development dates:", cut)
    print("Final-test dates:", len(dates) - cut)
    print(
        "Final test:",
        pd.Timestamp(dates[cut]).date(),
        "to",
        pd.Timestamp(dates[-1]).date()
    )


## Step 6 — Recreate the Same Expanding-Window Folds


In [ ]:
# ============================================================
# STEP 6: Recreate Expanding-Window Folds
# ============================================================

def build_market_folds(dev_dates, n_splits=4):

    n = len(dev_dates)

    test_size = max(
        8,
        int(np.floor(0.10 * n))
    )

    while n_splits * test_size >= int(0.60 * n):
        test_size -= 1

    if test_size < 5:
        raise ValueError(
            "Not enough development dates for four folds."
        )

    splitter = TimeSeriesSplit(
        n_splits=n_splits,
        test_size=test_size,
        gap=0
    )

    folds = []

    for fold_no, (tr_idx, va_idx) in enumerate(
        splitter.split(dev_dates),
        start=1
    ):
        folds.append({
            "Fold": fold_no,
            "Train_Dates": dev_dates[tr_idx],
            "Val_Dates": dev_dates[va_idx]
        })

    return folds


market_folds = {}

for market_id in MARKETS:

    folds = build_market_folds(
        market_splits[market_id]["Dev_Dates"],
        n_splits=4
    )

    market_folds[market_id] = folds

    print("\n", market_id)

    for fold in folds:
        print(
            f"Fold {fold['Fold']}:",
            pd.Timestamp(fold["Val_Dates"][0]).date(),
            "to",
            pd.Timestamp(fold["Val_Dates"][-1]).date()
        )


## Step 7 — Load the Already-Selected ML Hyperparameters

No tuning is repeated.


In [ ]:
# ============================================================
# STEP 7: Parse Previously Selected ML Hyperparameters
# ============================================================

best_ml_params = {}

for _, row in best_ml.iterrows():

    key = (
        row["Market_ID"],
        row["Model"],
        row["Specification"]
    )

    best_ml_params[key] = ast.literal_eval(
        row["Parameters"]
    )

print("Loaded parameter sets:", len(best_ml_params))

for key, value in list(best_ml_params.items())[:8]:
    print(key, "->", value)


## Step 8 — Event-Level Evaluation Function


In [ ]:
# ============================================================
# STEP 8: Evaluate Competitive Event Probabilities
# ============================================================

def evaluate_event_predictions(pred):

    eps = 1e-15
    rows = []

    for event_id, g in pred.groupby("Event_ID"):

        g = g.copy()

        probs = g["Pred_Prob"].to_numpy(dtype=float)
        probs = probs / probs.sum()

        y = g["Winner"].to_numpy(dtype=int)

        actual_idx = np.where(y == 1)[0][0]
        predicted_idx = int(np.argmax(probs))

        actual_winner = g.iloc[actual_idx]["Asset"]
        predicted_winner = g.iloc[predicted_idx]["Asset"]

        winner_prob = float(
            np.clip(
                probs[actual_idx],
                eps,
                1 - eps
            )
        )

        rows.append({
            "Market_ID": g["Market_ID"].iloc[0],
            "Event_ID": event_id,
            "Date": g["Date"].iloc[0],
            "Choice_Set_Size": len(g),
            "Actual_Winner": actual_winner,
            "Predicted_Winner": predicted_winner,
            "Winner_Probability": winner_prob,
            "Correct": int(
                actual_winner == predicted_winner
            ),
            "Log_Loss": -np.log(winner_prob),
            "Brier_Score": np.sum(
                (probs - y) ** 2
            )
        })

    return pd.DataFrame(rows)


## Step 9 — Conditional Logit Helper — Corrected Indexing

This version resets the test index and stores utility directly inside the prediction dataframe, avoiding the earlier `IndexError`.


In [ ]:
# ============================================================
# STEP 9: Corrected Conditional Logit Fit and Predict
# ============================================================

def fit_clogit_predict(
    train_df,
    test_df,
    features
):

    train_df = train_df.reset_index(
        drop=True
    ).copy()

    test_df = test_df.reset_index(
        drop=True
    ).copy()

    scaler = StandardScaler()

    X_train = scaler.fit_transform(
        train_df[features]
    )

    X_test = scaler.transform(
        test_df[features]
    )

    y_train = train_df[
        "Winner"
    ].astype(int).to_numpy()

    groups = train_df[
        "Event_ID"
    ].astype(str).to_numpy()

    model = ConditionalLogit(
        y_train,
        X_train,
        groups=groups
    )

    result = model.fit(
        method="bfgs",
        maxiter=1000,
        disp=False
    )

    pred = test_df[
        [
            "Market_ID",
            "Event_ID",
            "Date",
            "Asset",
            "Winner"
        ]
    ].copy()

    pred["Utility"] = (
        X_test
        @ np.asarray(result.params)
    )

    pred["Pred_Prob"] = (
        pred
        .groupby("Event_ID")["Utility"]
        .transform(
            lambda x: softmax(
                x.to_numpy()
            )
        )
    )

    return pred


## Step 10 — Traditional ML Helpers


In [ ]:
# ============================================================
# STEP 10A: Build Previously Selected ML Models
# ============================================================

def build_model(model_name, params):

    if model_name == "Logistic_Regression":

        return Pipeline([
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                LogisticRegression(
                    C=params["C"],
                    penalty="l2",
                    solver="lbfgs",
                    max_iter=2000,
                    random_state=42
                )
            )
        ])

    if model_name == "Random_Forest":

        return RandomForestClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_samples_leaf=params["min_samples_leaf"],
            max_features=params["max_features"],
            random_state=42,
            n_jobs=-1
        )

    if model_name == "XGBoost":

        return XGBClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            learning_rate=params["learning_rate"],
            subsample=params["subsample"],
            colsample_bytree=params["colsample_bytree"],
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
            tree_method="hist",
            verbosity=0
        )

    raise ValueError("Unknown model.")


In [ ]:
# ============================================================
# STEP 10B: Convert ML Winner Scores to Event Probabilities
# ============================================================

def ml_event_predictions(
    model,
    test_df,
    features
):

    test_df = test_df.reset_index(
        drop=True
    ).copy()

    raw_prob = model.predict_proba(
        test_df[features]
    )[:, 1]

    pred = test_df[
        [
            "Market_ID",
            "Event_ID",
            "Date",
            "Asset",
            "Winner"
        ]
    ].copy()

    pred["Raw_Winner_Prob"] = raw_prob
    pred["Pred_Prob"] = np.nan

    for event_id, idx in pred.groupby(
        "Event_ID"
    ).groups.items():

        idx = list(idx)

        probs = pred.loc[
            idx,
            "Raw_Winner_Prob"
        ].to_numpy(dtype=float)

        probs = np.clip(
            probs,
            1e-15,
            None
        )

        pred.loc[
            idx,
            "Pred_Prob"
        ] = probs / probs.sum()

    return pred


## Step 11 — Benchmark Helper 1: Raw Polymarket Favourite

This uses `PM_Prob_Lag1` directly.

No model is fitted.

Because these probabilities already represent the competitive Polymarket probabilities, they are simply normalized within each Event ID as a safety check.


In [ ]:
# ============================================================
# STEP 11: Raw Polymarket Favourite Benchmark
# ============================================================

def polymarket_benchmark(test_df):

    test_df = test_df.reset_index(
        drop=True
    ).copy()

    pred = test_df[
        [
            "Market_ID",
            "Event_ID",
            "Date",
            "Asset",
            "Winner",
            "PM_Prob_Lag1"
        ]
    ].copy()

    pred["Pred_Prob"] = np.nan

    for event_id, idx in pred.groupby(
        "Event_ID"
    ).groups.items():

        idx = list(idx)

        probs = pred.loc[
            idx,
            "PM_Prob_Lag1"
        ].to_numpy(dtype=float)

        probs = np.clip(
            probs,
            1e-15,
            None
        )

        pred.loc[
            idx,
            "Pred_Prob"
        ] = probs / probs.sum()

    return pred


## Step 12 — Benchmark Helper 2: Training Winner-Frequency

For each training period, this benchmark calculates the historical winner share of each asset.

Example:

- Bitcoin won 40% of training events
- Gold won 60%

Then every test event receives:

- Bitcoin = 0.40
- Gold = 0.60

This benchmark is leakage-safe because the test outcomes are never used to construct the probabilities.


In [ ]:
# ============================================================
# STEP 12: Training Winner-Frequency Benchmark
# ============================================================

def training_frequency_benchmark(
    train_df,
    test_df
):

    train_winners = train_df[
        train_df["Winner"] == 1
    ].copy()

    winner_counts = (
        train_winners[
            "Asset"
        ]
        .value_counts()
    )

    all_assets = (
        test_df[
            "Asset"
        ]
        .unique()
        .tolist()
    )

    total_events = (
        train_df[
            "Event_ID"
        ].nunique()
    )

    frequency = {}

    for asset in all_assets:
        frequency[asset] = (
            winner_counts.get(
                asset,
                0
            )
            / total_events
        )

    pred = test_df[
        [
            "Market_ID",
            "Event_ID",
            "Date",
            "Asset",
            "Winner"
        ]
    ].reset_index(
        drop=True
    ).copy()

    pred[
        "Training_Winner_Frequency"
    ] = pred[
        "Asset"
    ].map(frequency)

    pred[
        "Pred_Prob"
    ] = np.nan

    for event_id, idx in pred.groupby(
        "Event_ID"
    ).groups.items():

        idx = list(idx)

        probs = pred.loc[
            idx,
            "Training_Winner_Frequency"
        ].to_numpy(dtype=float)

        probs = np.clip(
            probs,
            1e-15,
            None
        )

        pred.loc[
            idx,
            "Pred_Prob"
        ] = probs / probs.sum()

    return pred


## Step 13 — Generate Expanding-Window OOS Predictions for the Four Models

No model selection or hyperparameter tuning occurs here.

Only the already-selected specifications are refitted on each existing training window.


In [ ]:
# ============================================================
# STEP 13: Regenerate Model OOS Validation Predictions
# ============================================================

cv_model_tables = []

for market_id in MARKETS:

    market_df = df[
        df["Market_ID"] == market_id
    ].copy()

    folds = market_folds[
        market_id
    ]

    print(
        "\nGenerating OOS model predictions:",
        market_id
    )

    # -------------------------
    # Conditional Logit
    # -------------------------

    for specification, features in [
        ("Baseline", BASELINE_FEATURES),
        ("Enhanced", CL_ENHANCED_FEATURES)
    ]:

        for fold in folds:

            train_df = market_df[
                market_df["Date"].isin(
                    fold["Train_Dates"]
                )
            ].copy()

            val_df = market_df[
                market_df["Date"].isin(
                    fold["Val_Dates"]
                )
            ].copy()

            pred = fit_clogit_predict(
                train_df,
                val_df,
                features
            )

            events = evaluate_event_predictions(
                pred
            )

            events["Model"] = "Conditional_Logit"
            events["Specification"] = specification
            events["OOS_Source"] = "Expanding_Validation"
            events["Fold"] = fold["Fold"]
            events["OOS_Block"] = (
                "Fold_"
                + str(fold["Fold"])
            )

            cv_model_tables.append(
                events
            )

    # -------------------------
    # Traditional ML
    # -------------------------

    for model_name in ML_MODELS:

        for specification, features in [
            ("Baseline", BASELINE_FEATURES),
            ("Enhanced", ML_ENHANCED_FEATURES)
        ]:

            params = best_ml_params[
                (
                    market_id,
                    model_name,
                    specification
                )
            ]

            for fold in folds:

                train_df = market_df[
                    market_df["Date"].isin(
                        fold["Train_Dates"]
                    )
                ].copy()

                val_df = market_df[
                    market_df["Date"].isin(
                        fold["Val_Dates"]
                    )
                ].copy()

                model = build_model(
                    model_name,
                    params
                )

                model.fit(
                    train_df[features],
                    train_df["Winner"].astype(int)
                )

                pred = ml_event_predictions(
                    model,
                    val_df,
                    features
                )

                events = evaluate_event_predictions(
                    pred
                )

                events["Model"] = model_name
                events["Specification"] = specification
                events["OOS_Source"] = "Expanding_Validation"
                events["Fold"] = fold["Fold"]
                events["OOS_Block"] = (
                    "Fold_"
                    + str(fold["Fold"])
                )

                cv_model_tables.append(
                    events
                )

cv_model_oos = pd.concat(
    cv_model_tables,
    ignore_index=True
)

print(
    "Validation model prediction rows:",
    len(cv_model_oos)
)

display(cv_model_oos.head())


## Step 14 — Import the Previously Saved Final-Holdout Model Predictions

The final model predictions are not recomputed.

They are imported from the previous market-specific robustness workbook.


In [ ]:
# ============================================================
# STEP 14: Prepare Saved Final-Holdout Model Predictions
# ============================================================

final_model_oos = (
    saved_final_predictions.copy()
)

final_model_oos["OOS_Source"] = "Final_Holdout"
final_model_oos["Fold"] = np.nan
final_model_oos["OOS_Block"] = "Final_Holdout"

needed = [
    "Market_ID",
    "Event_ID",
    "Date",
    "Choice_Set_Size",
    "Actual_Winner",
    "Predicted_Winner",
    "Winner_Probability",
    "Correct",
    "Log_Loss",
    "Brier_Score",
    "Model",
    "Specification",
    "OOS_Source",
    "Fold",
    "OOS_Block"
]

final_model_oos = final_model_oos[
    needed
]

cv_model_oos = cv_model_oos[
    needed
]

all_model_oos = pd.concat(
    [
        cv_model_oos,
        final_model_oos
    ],
    ignore_index=True
)

print(
    "Combined model OOS rows:",
    len(all_model_oos)
)

print(
    all_model_oos[
        "OOS_Source"
    ].value_counts()
)


## Step 15 — Generate Both Benchmarks for Every OOS Period

Both validation folds and final holdouts are evaluated.

The training-frequency benchmark is reconstructed separately for each fold/test period using only its corresponding training data.


In [ ]:
# ============================================================
# STEP 15: Generate OOS Benchmark Predictions
# ============================================================

benchmark_tables = []

for market_id in MARKETS:

    market_df = df[
        df["Market_ID"] == market_id
    ].copy()

    # ------------------------------------
    # Expanding-window validation folds
    # ------------------------------------

    for fold in market_folds[market_id]:

        train_df = market_df[
            market_df["Date"].isin(
                fold["Train_Dates"]
            )
        ].copy()

        val_df = market_df[
            market_df["Date"].isin(
                fold["Val_Dates"]
            )
        ].copy()

        # Raw Polymarket benchmark
        pm_pred = polymarket_benchmark(
            val_df
        )

        pm_events = evaluate_event_predictions(
            pm_pred
        )

        pm_events["Benchmark"] = "Raw_PM_Favourite"
        pm_events["OOS_Source"] = "Expanding_Validation"
        pm_events["Fold"] = fold["Fold"]
        pm_events["OOS_Block"] = (
            "Fold_"
            + str(fold["Fold"])
        )

        benchmark_tables.append(
            pm_events
        )

        # Training-frequency benchmark
        freq_pred = training_frequency_benchmark(
            train_df,
            val_df
        )

        freq_events = evaluate_event_predictions(
            freq_pred
        )

        freq_events["Benchmark"] = "Training_Winner_Frequency"
        freq_events["OOS_Source"] = "Expanding_Validation"
        freq_events["Fold"] = fold["Fold"]
        freq_events["OOS_Block"] = (
            "Fold_"
            + str(fold["Fold"])
        )

        benchmark_tables.append(
            freq_events
        )

    # ------------------------------------
    # Final holdout
    # ------------------------------------

    dev_dates = market_splits[
        market_id
    ]["Dev_Dates"]

    test_dates = market_splits[
        market_id
    ]["Test_Dates"]

    train_df = market_df[
        market_df["Date"].isin(
            dev_dates
        )
    ].copy()

    test_df = market_df[
        market_df["Date"].isin(
            test_dates
        )
    ].copy()

    # Raw PM
    pm_pred = polymarket_benchmark(
        test_df
    )

    pm_events = evaluate_event_predictions(
        pm_pred
    )

    pm_events["Benchmark"] = "Raw_PM_Favourite"
    pm_events["OOS_Source"] = "Final_Holdout"
    pm_events["Fold"] = np.nan
    pm_events["OOS_Block"] = "Final_Holdout"

    benchmark_tables.append(
        pm_events
    )

    # Training frequency
    freq_pred = training_frequency_benchmark(
        train_df,
        test_df
    )

    freq_events = evaluate_event_predictions(
        freq_pred
    )

    freq_events["Benchmark"] = "Training_Winner_Frequency"
    freq_events["OOS_Source"] = "Final_Holdout"
    freq_events["Fold"] = np.nan
    freq_events["OOS_Block"] = "Final_Holdout"

    benchmark_tables.append(
        freq_events
    )

benchmark_oos = pd.concat(
    benchmark_tables,
    ignore_index=True
)

print(
    "Benchmark OOS rows:",
    len(benchmark_oos)
)

display(benchmark_oos.head())


## Step 16 — Add Event Time-to-Resolution and Polymarket Confidence

For each event we retain:

- `Days_to_Resolution`
- `Days_to_Resolution_Years`
- `PM_Max_Prob`: highest lagged Polymarket probability in the event
- `PM_Favourite_Asset`
- `PM_Normalized_Entropy`: uncertainty of the Polymarket distribution

Interpretation of normalized entropy:

- 0 = highly certain / concentrated
- 1 = maximally uncertain / uniform

These are descriptive diagnostics only. They are not new model predictors.


In [ ]:
# ============================================================
# STEP 16: Create Event Horizon and Polymarket Confidence Data
# ============================================================

confidence_rows = []

for (
    market_id,
    event_id,
    date
), g in df.groupby(
    [
        "Market_ID",
        "Event_ID",
        "Date"
    ]
):

    probs = g[
        "PM_Prob_Lag1"
    ].to_numpy(dtype=float)

    probs = np.clip(
        probs,
        1e-15,
        None
    )

    probs = probs / probs.sum()

    max_idx = int(
        np.argmax(probs)
    )

    entropy = -np.sum(
        probs * np.log(probs)
    )

    normalized_entropy = (
        entropy / np.log(len(probs))
    )

    confidence_rows.append({
        "Market_ID":
            market_id,
        "Event_ID":
            event_id,
        "Date":
            date,
        "Days_to_Resolution":
            g[
                "Days_to_Resolution"
            ].iloc[0],
        "Days_to_Resolution_Years":
            g[
                "Days_to_Resolution_Years"
            ].iloc[0],
        "PM_Max_Prob":
            probs[max_idx],
        "PM_Favourite_Asset":
            g.iloc[max_idx][
                "Asset"
            ],
        "PM_Normalized_Entropy":
            normalized_entropy
    })

event_context = pd.DataFrame(
    confidence_rows
)

all_model_oos = all_model_oos.merge(
    event_context,
    on=[
        "Market_ID",
        "Event_ID",
        "Date"
    ],
    how="left"
)

benchmark_oos = benchmark_oos.merge(
    event_context,
    on=[
        "Market_ID",
        "Event_ID",
        "Date"
    ],
    how="left"
)

print("Event context merged successfully.")


## Step 17 — Correctly Pair Baseline and Enhanced OOS Predictions

The key fix is here:

`OOS_Block` is used instead of raw `Fold`.

Final-holdout observations therefore remain in the paired dataset.


In [ ]:
# ============================================================
# STEP 17: Pair Baseline and Enhanced — CORRECTED
# ============================================================

pair_index = [
    "Market_ID",
    "Event_ID",
    "Date",
    "Model",
    "OOS_Source",
    "OOS_Block",
    "Days_to_Resolution",
    "Days_to_Resolution_Years",
    "PM_Max_Prob",
    "PM_Favourite_Asset",
    "PM_Normalized_Entropy"
]

paired = all_model_oos.pivot_table(
    index=pair_index,
    columns="Specification",
    values=[
        "Log_Loss",
        "Brier_Score",
        "Correct"
    ],
    aggfunc="first",
    dropna=True
).reset_index()

paired.columns = [
    "_".join(
        [
            str(x)
            for x in col
            if str(x) != ""
        ]
    )
    if isinstance(col, tuple)
    else col
    for col in paired.columns
]

paired = paired.rename(
    columns={
        "Log_Loss_Baseline":
            "Baseline_Log_Loss",
        "Log_Loss_Enhanced":
            "Enhanced_Log_Loss",
        "Brier_Score_Baseline":
            "Baseline_Brier",
        "Brier_Score_Enhanced":
            "Enhanced_Brier",
        "Correct_Baseline":
            "Baseline_Correct",
        "Correct_Enhanced":
            "Enhanced_Correct"
    }
)

paired["LogLoss_Gain"] = (
    paired["Baseline_Log_Loss"]
    -
    paired["Enhanced_Log_Loss"]
)

paired["Brier_Gain"] = (
    paired["Baseline_Brier"]
    -
    paired["Enhanced_Brier"]
)

paired["Accuracy_Gain"] = (
    paired["Enhanced_Correct"]
    -
    paired["Baseline_Correct"]
)

# Keep only rows where both specifications were found
paired = paired.dropna(
    subset=[
        "Baseline_Log_Loss",
        "Enhanced_Log_Loss"
    ]
).reset_index(drop=True)

print(
    "Paired model-event rows:",
    len(paired)
)

print("\nPaired rows by OOS source:")
print(
    paired[
        "OOS_Source"
    ].value_counts()
)

print("\nFinal-holdout paired rows:")
print(
    len(
        paired[
            paired["OOS_Source"]
            == "Final_Holdout"
        ]
    )
)

assert (
    paired[
        paired["OOS_Source"]
        == "Final_Holdout"
    ].shape[0] > 0
), "ERROR: Final holdout is still missing."

print(
    "\nCORRECTION PASSED: final holdout is retained."
)


## Step 18 — OOS Coverage Check


In [ ]:
# ============================================================
# STEP 18: OOS Coverage Summary
# ============================================================

coverage = (
    paired.groupby(
        "Market_ID"
    )
    .agg(
        Unique_OOS_Events=(
            "Event_ID",
            "nunique"
        ),
        First_OOS_Date=(
            "Date",
            "min"
        ),
        Last_OOS_Date=(
            "Date",
            "max"
        ),
        Max_Days_to_Resolution=(
            "Days_to_Resolution",
            "max"
        ),
        Min_Days_to_Resolution=(
            "Days_to_Resolution",
            "min"
        )
    )
    .reset_index()
)

display(coverage)


## Step 19 — Final-Holdout Model Results with Benchmark Comparison

This is the most important descriptive table.

For each market, it shows:

- Baseline model
- Enhanced model
- raw Polymarket favourite
- training winner-frequency benchmark


In [ ]:
# ============================================================
# STEP 19: Final Holdout Models vs Benchmarks
# ============================================================

final_model_summary = (
    all_model_oos[
        all_model_oos[
            "OOS_Source"
        ] == "Final_Holdout"
    ]
    .groupby(
        [
            "Market_ID",
            "Model",
            "Specification"
        ]
    )
    .agg(
        Events=(
            "Event_ID",
            "count"
        ),
        Log_Loss=(
            "Log_Loss",
            "mean"
        ),
        Brier_Score=(
            "Brier_Score",
            "mean"
        ),
        Accuracy=(
            "Correct",
            "mean"
        ),
        Mean_PM_Max_Prob=(
            "PM_Max_Prob",
            "mean"
        ),
        Mean_Days_to_Resolution=(
            "Days_to_Resolution",
            "mean"
        )
    )
    .reset_index()
)

final_benchmark_summary = (
    benchmark_oos[
        benchmark_oos[
            "OOS_Source"
        ] == "Final_Holdout"
    ]
    .groupby(
        [
            "Market_ID",
            "Benchmark"
        ]
    )
    .agg(
        Events=(
            "Event_ID",
            "count"
        ),
        Log_Loss=(
            "Log_Loss",
            "mean"
        ),
        Brier_Score=(
            "Brier_Score",
            "mean"
        ),
        Accuracy=(
            "Correct",
            "mean"
        ),
        Mean_PM_Max_Prob=(
            "PM_Max_Prob",
            "mean"
        ),
        Mean_Days_to_Resolution=(
            "Days_to_Resolution",
            "mean"
        )
    )
    .reset_index()
)

print("FINAL MODEL RESULTS")
display(final_model_summary)

print("\nFINAL BENCHMARK RESULTS")
display(final_benchmark_summary)


## Step 20 — BTC–Gold 2025 Favourite Diagnostic

This directly checks the concern that the apparent 2025 improvement may mainly reflect Polymarket becoming almost certain about Gold.

We report:

- how often the raw Polymarket favourite is Gold;
- the average maximum Polymarket probability;
- raw favourite accuracy;
- Enhanced-model accuracy;
- training-frequency benchmark accuracy.


In [ ]:
# ============================================================
# STEP 20: BTC-Gold 2025 Near-Resolution Favourite Diagnostic
# ============================================================

btc25_context = event_context[
    event_context[
        "Market_ID"
    ] == "BTC_GOLD_2025"
].copy()

btc25_final_ids = set(
    paired[
        (
            paired["Market_ID"]
            == "BTC_GOLD_2025"
        )
        &
        (
            paired["OOS_Source"]
            == "Final_Holdout"
        )
    ]["Event_ID"].unique()
)

btc25_final_context = btc25_context[
    btc25_context[
        "Event_ID"
    ].isin(
        btc25_final_ids
    )
]

favourite_distribution = (
    btc25_final_context[
        "PM_Favourite_Asset"
    ]
    .value_counts(
        normalize=True
    )
    .rename(
        "Share"
    )
    .reset_index()
)

favourite_distribution.columns = [
    "PM_Favourite_Asset",
    "Share"
]

print(
    "BTC-Gold 2025 final holdout:"
)

print(
    "Mean days to resolution:",
    btc25_final_context[
        "Days_to_Resolution"
    ].mean()
)

print(
    "Mean Polymarket max probability:",
    btc25_final_context[
        "PM_Max_Prob"
    ].mean()
)

print(
    "\nPolymarket favourite distribution:"
)

display(favourite_distribution)

btc25_bench = final_benchmark_summary[
    final_benchmark_summary[
        "Market_ID"
    ] == "BTC_GOLD_2025"
]

print(
    "\nBTC-Gold 2025 final benchmark performance:"
)

display(btc25_bench)

btc25_models = final_model_summary[
    final_model_summary[
        "Market_ID"
    ] == "BTC_GOLD_2025"
]

print(
    "\nBTC-Gold 2025 final model performance:"
)

display(btc25_models)


## Step 21 — Create Fixed Time-to-Resolution Bands

Descriptive bands:

- ≤30 days
- 31–90 days
- 91–180 days
- >180 days


In [ ]:
# ============================================================
# STEP 21: Fixed Resolution Bands
# ============================================================

paired["Resolution_Band"] = pd.cut(
    paired["Days_to_Resolution"],
    bins=[
        -np.inf,
        30,
        90,
        180,
        np.inf
    ],
    labels=[
        "≤30 days",
        "31–90 days",
        "91–180 days",
        ">180 days"
    ]
)

band_counts = (
    paired.groupby(
        [
            "Market_ID",
            "Resolution_Band"
        ],
        observed=True
    )[
        "Event_ID"
    ]
    .nunique()
    .reset_index(
        name="Unique_Events"
    )
)

display(band_counts)


## Step 22 — Polymarket Incremental Gain by Horizon Band and Model


In [ ]:
# ============================================================
# STEP 22: Horizon Band Gain Summary
# ============================================================

band_summary_by_model = (
    paired.groupby(
        [
            "Market_ID",
            "Model",
            "Resolution_Band"
        ],
        observed=True
    )
    .agg(
        Observations=(
            "Event_ID",
            "count"
        ),
        Mean_LogLoss_Gain=(
            "LogLoss_Gain",
            "mean"
        ),
        Mean_Brier_Gain=(
            "Brier_Gain",
            "mean"
        ),
        Mean_Accuracy_Gain=(
            "Accuracy_Gain",
            "mean"
        ),
        Mean_PM_Max_Prob=(
            "PM_Max_Prob",
            "mean"
        )
    )
    .reset_index()
)

display(
    band_summary_by_model.sort_values(
        [
            "Market_ID",
            "Model",
            "Resolution_Band"
        ]
    )
)


## Step 23 — Architecture-Averaged Event-Level Gain

The same event is predicted by four models.

To avoid treating four model architectures as four independent financial events, we average the Baseline-vs-Enhanced gain across model architectures for each Event ID.


In [ ]:
# ============================================================
# STEP 23: Average Incremental Gain Across Models
# ============================================================

event_average = (
    paired.groupby(
        [
            "Market_ID",
            "Event_ID",
            "Date",
            "OOS_Source",
            "OOS_Block",
            "Days_to_Resolution",
            "Days_to_Resolution_Years",
            "PM_Max_Prob",
            "PM_Favourite_Asset",
            "PM_Normalized_Entropy"
        ],
        as_index=False
    )
    .agg(
        Mean_LogLoss_Gain_Across_Models=(
            "LogLoss_Gain",
            "mean"
        ),
        Mean_Brier_Gain_Across_Models=(
            "Brier_Gain",
            "mean"
        ),
        Mean_Accuracy_Gain_Across_Models=(
            "Accuracy_Gain",
            "mean"
        ),
        Models_Available=(
            "Model",
            "nunique"
        )
    )
)

print(
    "Architecture-averaged OOS events:",
    len(event_average)
)

print(
    event_average[
        "Models_Available"
    ].value_counts()
)

display(event_average.head())


## Step 24 — Near vs Far Within Each Market

Each market is divided using its own median out-of-sample time to resolution.

This avoids comparing different contracts only because one contract is generally shorter-horizon than another.


In [ ]:
# ============================================================
# STEP 24: Within-Market Near vs Far Analysis
# ============================================================

event_average[
    "Within_Market_Horizon_Group"
] = None

median_days_table = []

for market_id, g in event_average.groupby(
    "Market_ID"
):

    median_days = g[
        "Days_to_Resolution"
    ].median()

    median_days_table.append({
        "Market_ID":
            market_id,
        "Median_OOS_Days_to_Resolution":
            median_days
    })

    idx = (
        event_average[
            "Market_ID"
        ] == market_id
    )

    event_average.loc[
        idx,
        "Within_Market_Horizon_Group"
    ] = np.where(
        event_average.loc[
            idx,
            "Days_to_Resolution"
        ] <= median_days,
        "Nearer_to_Resolution",
        "Farther_from_Resolution"
    )

median_days_df = pd.DataFrame(
    median_days_table
)

near_far_summary = (
    event_average.groupby(
        [
            "Market_ID",
            "Within_Market_Horizon_Group"
        ]
    )
    .agg(
        Events=(
            "Event_ID",
            "count"
        ),
        Mean_Days_to_Resolution=(
            "Days_to_Resolution",
            "mean"
        ),
        Mean_PM_Max_Prob=(
            "PM_Max_Prob",
            "mean"
        ),
        Mean_PM_Entropy=(
            "PM_Normalized_Entropy",
            "mean"
        ),
        Mean_LogLoss_Gain=(
            "Mean_LogLoss_Gain_Across_Models",
            "mean"
        ),
        Mean_Brier_Gain=(
            "Mean_Brier_Gain_Across_Models",
            "mean"
        ),
        Mean_Accuracy_Gain=(
            "Mean_Accuracy_Gain_Across_Models",
            "mean"
        )
    )
    .reset_index()
)

print("Within-market median horizons:")
display(median_days_df)

print("\nNear vs far results:")
display(near_far_summary)


## Step 25 — Continuous Time-to-Resolution Test Within Each Market

We test whether architecture-averaged Polymarket gain changes continuously with days to resolution.

Interpretation:

- Negative coefficient = gain tends to be stronger nearer resolution
- Positive coefficient = gain tends to be stronger farther from resolution

Newey–West HAC standard errors are used because the event sequence is time ordered.


In [ ]:
# ============================================================
# STEP 25: Continuous Horizon Tests
# ============================================================

continuous_rows = []

for market_id, g in event_average.groupby(
    "Market_ID"
):

    g = g.sort_values(
        "Date"
    ).copy()

    x = g[
        "Days_to_Resolution_Years"
    ].astype(float)

    y = g[
        "Mean_LogLoss_Gain_Across_Models"
    ].astype(float)

    rho, rho_p = spearmanr(
        x,
        y
    )

    X = sm.add_constant(
        x.to_numpy()
    )

    fit = sm.OLS(
        y.to_numpy(),
        X
    ).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": 3
        }
    )

    continuous_rows.append({
        "Market_ID":
            market_id,
        "OOS_Events":
            len(g),
        "Spearman_Rho":
            rho,
        "Spearman_p":
            rho_p,
        "Days_to_Resolution_Coefficient":
            fit.params[1],
        "HAC_Std_Error":
            fit.bse[1],
        "t_Statistic":
            fit.tvalues[1],
        "HAC_p_Value":
            fit.pvalues[1]
    })

continuous_tests = pd.DataFrame(
    continuous_rows
)

display(continuous_tests)

print(
    "\nNegative coefficient = stronger Polymarket gain nearer resolution."
)


## Step 26 — Model-Specific Horizon Correlations


In [ ]:
# ============================================================
# STEP 26: Model-Specific Horizon Correlations
# ============================================================

model_corr_rows = []

for (
    market_id,
    model_name
), g in paired.groupby(
    [
        "Market_ID",
        "Model"
    ]
):

    rho, p_value = spearmanr(
        g[
            "Days_to_Resolution"
        ],
        g[
            "LogLoss_Gain"
        ]
    )

    model_corr_rows.append({
        "Market_ID":
            market_id,
        "Model":
            model_name,
        "OOS_Events":
            len(g),
        "Spearman_Rho":
            rho,
        "p_Value":
            p_value
    })

model_correlations = pd.DataFrame(
    model_corr_rows
)

display(
    model_correlations.sort_values(
        [
            "Market_ID",
            "Model"
        ]
    )
)


## Step 27 — Does Polymarket Confidence Increase Near Resolution?

This is descriptive evidence relevant to the 2025 concern.

We test the relationship between:

- days to resolution;
- maximum Polymarket probability;
- normalized entropy.

Expected pattern for increasing certainty near resolution:

- `PM_Max_Prob` increases as days decrease;
- entropy decreases as days decrease.


In [ ]:
# ============================================================
# STEP 27: Polymarket Certainty vs Time to Resolution
# ============================================================

certainty_rows = []

for market_id, g in event_average.groupby(
    "Market_ID"
):

    rho_max, p_max = spearmanr(
        g[
            "Days_to_Resolution"
        ],
        g[
            "PM_Max_Prob"
        ]
    )

    rho_entropy, p_entropy = spearmanr(
        g[
            "Days_to_Resolution"
        ],
        g[
            "PM_Normalized_Entropy"
        ]
    )

    certainty_rows.append({
        "Market_ID":
            market_id,
        "Days_vs_MaxProb_Rho":
            rho_max,
        "Days_vs_MaxProb_p":
            p_max,
        "Days_vs_Entropy_Rho":
            rho_entropy,
        "Days_vs_Entropy_p":
            p_entropy
    })

certainty_tests = pd.DataFrame(
    certainty_rows
)

display(certainty_tests)


## Step 28 — Final-Holdout-Only Incremental Gain

This keeps the strict untouched test evidence separate from the combined expanding-window analysis.


In [ ]:
# ============================================================
# STEP 28: Final-Holdout-Only Polymarket Gain
# ============================================================

final_only = paired[
    paired[
        "OOS_Source"
    ] == "Final_Holdout"
].copy()

final_only_summary = (
    final_only.groupby(
        [
            "Market_ID",
            "Model"
        ]
    )
    .agg(
        Events=(
            "Event_ID",
            "count"
        ),
        Mean_Days_to_Resolution=(
            "Days_to_Resolution",
            "mean"
        ),
        Mean_PM_Max_Prob=(
            "PM_Max_Prob",
            "mean"
        ),
        Mean_LogLoss_Gain=(
            "LogLoss_Gain",
            "mean"
        ),
        Mean_Brier_Gain=(
            "Brier_Gain",
            "mean"
        ),
        Mean_Accuracy_Gain=(
            "Accuracy_Gain",
            "mean"
        )
    )
    .reset_index()
)

display(final_only_summary)

assert len(final_only_summary) > 0,     "ERROR: Final holdout summary is empty."


## Step 29 — Visualize Incremental Log-Loss Gain vs Time to Resolution

Above zero = Enhanced model better.

Below zero = Baseline better.


In [ ]:
# ============================================================
# STEP 29: Plot Polymarket Gain vs Days to Resolution
# ============================================================

for market_id in MARKETS:

    g = event_average[
        event_average[
            "Market_ID"
        ] == market_id
    ].sort_values(
        "Days_to_Resolution"
    )

    plt.figure(
        figsize=(8, 5)
    )

    plt.scatter(
        g[
            "Days_to_Resolution"
        ],
        g[
            "Mean_LogLoss_Gain_Across_Models"
        ]
    )

    plt.axhline(
        0,
        linewidth=1
    )

    plt.xlabel(
        "Days to Resolution"
    )

    plt.ylabel(
        "Baseline Log Loss - Enhanced Log Loss"
    )

    plt.title(
        f"{market_id}: "
        "Incremental Polymarket Value vs Time to Resolution"
    )

    plt.grid(
        alpha=0.2
    )

    plt.show()


## Step 30 — Compact Felipe Summary

This table combines:

- time-to-resolution direction;
- statistical evidence;
- final raw Polymarket favourite performance;
- final training-frequency benchmark performance.

The interpretation should remain cautious when statistical evidence is weak.


In [ ]:
# ============================================================
# STEP 30: Compact Supervisor Summary
# ============================================================

felipe_rows = []

for _, r in continuous_tests.iterrows():

    market_id = r[
        "Market_ID"
    ]

    coef = r[
        "Days_to_Resolution_Coefficient"
    ]

    pval = r[
        "HAC_p_Value"
    ]

    if coef < 0:
        direction = (
            "Polymarket gain tends "
            "to strengthen nearer resolution"
        )
    else:
        direction = (
            "No near-resolution "
            "strengthening pattern"
        )

    if pval < 0.05:
        evidence = "Significant at 5%"
    elif pval < 0.10:
        evidence = "Suggestive at 10%"
    else:
        evidence = "Not statistically significant"

    pm_row = final_benchmark_summary[
        (
            final_benchmark_summary[
                "Market_ID"
            ] == market_id
        )
        &
        (
            final_benchmark_summary[
                "Benchmark"
            ] == "Raw_PM_Favourite"
        )
    ]

    freq_row = final_benchmark_summary[
        (
            final_benchmark_summary[
                "Market_ID"
            ] == market_id
        )
        &
        (
            final_benchmark_summary[
                "Benchmark"
            ] == "Training_Winner_Frequency"
        )
    ]

    pm_acc = (
        pm_row[
            "Accuracy"
        ].iloc[0]
        if len(pm_row)
        else np.nan
    )

    freq_acc = (
        freq_row[
            "Accuracy"
        ].iloc[0]
        if len(freq_row)
        else np.nan
    )

    felipe_rows.append({
        "Market":
            market_id,
        "OOS Events":
            int(
                r[
                    "OOS_Events"
                ]
            ),
        "Horizon Coefficient":
            coef,
        "p-value":
            pval,
        "Direction":
            direction,
        "Evidence":
            evidence,
        "Final Raw PM Favourite Accuracy":
            pm_acc,
        "Final Training-Frequency Accuracy":
            freq_acc
    })

felipe_summary = pd.DataFrame(
    felipe_rows
)

display(felipe_summary)


## Step 31 — Save the Corrected Robustness Results

The output includes:

- all model OOS predictions;
- all benchmark OOS predictions;
- corrected paired Baseline/Enhanced gains;
- final-holdout model results;
- final-holdout benchmark results;
- BTC–Gold 2025 favourite diagnostic;
- horizon-band summaries;
- architecture-averaged gains;
- near-vs-far analysis;
- continuous horizon tests;
- model-specific correlations;
- Polymarket certainty analysis;
- final-holdout-only gain;
- compact Felipe summary.


In [ ]:
# ============================================================
# STEP 31: Save Corrected Results Workbook
# ============================================================

OUTPUT_FILE = (
    "/content/"
    "Time_to_Resolution_CORRECTED_with_Benchmarks_Results.xlsx"
)

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    all_model_oos.to_excel(
        writer,
        sheet_name="All_Model_OOS",
        index=False
    )

    benchmark_oos.to_excel(
        writer,
        sheet_name="All_Benchmark_OOS",
        index=False
    )

    paired.to_excel(
        writer,
        sheet_name="Paired_Model_Gains",
        index=False
    )

    coverage.to_excel(
        writer,
        sheet_name="OOS_Coverage",
        index=False
    )

    final_model_summary.to_excel(
        writer,
        sheet_name="Final_Model_Results",
        index=False
    )

    final_benchmark_summary.to_excel(
        writer,
        sheet_name="Final_Benchmarks",
        index=False
    )

    favourite_distribution.to_excel(
        writer,
        sheet_name="BTC25_Favourite_Share",
        index=False
    )

    band_counts.to_excel(
        writer,
        sheet_name="Horizon_Band_Counts",
        index=False
    )

    band_summary_by_model.to_excel(
        writer,
        sheet_name="Horizon_By_Model",
        index=False
    )

    event_average.to_excel(
        writer,
        sheet_name="Event_Average_Gains",
        index=False
    )

    median_days_df.to_excel(
        writer,
        sheet_name="Market_Median_Horizon",
        index=False
    )

    near_far_summary.to_excel(
        writer,
        sheet_name="Near_vs_Far",
        index=False
    )

    continuous_tests.to_excel(
        writer,
        sheet_name="Continuous_Horizon_Tests",
        index=False
    )

    model_correlations.to_excel(
        writer,
        sheet_name="Model_Horizon_Correlations",
        index=False
    )

    certainty_tests.to_excel(
        writer,
        sheet_name="PM_Certainty_Tests",
        index=False
    )

    final_only_summary.to_excel(
        writer,
        sheet_name="Final_Holdout_Only",
        index=False
    )

    felipe_summary.to_excel(
        writer,
        sheet_name="Felipe_Summary",
        index=False
    )

    feature_set_table.to_excel(
        writer,
        sheet_name="Feature_Set_Definitions",
        index=False
    )

print("Saved successfully:")
print(OUTPUT_FILE)


## Step 32 — Download the Corrected Results


In [ ]:
# ============================================================
# STEP 32: Download Results Workbook
# ============================================================

from google.colab import files

files.download(
    "/content/"
    "Time_to_Resolution_CORRECTED_with_Benchmarks_Results.xlsx"
)
